# Impedance ML Engineering — Session 05
## Model repeated measurements and temporal change
**Core: 60 minutes · Statistics · Optional extension: 20–40 minutes**

## Why this matters
How much variation is between patients versus within patients, and how does telemetry change over follow-up?

Build the judgment to implement, review, and challenge data and modeling workflows in an ML engineering role.

## Learning objectives
- Visualize individual trajectories and protocol overlap.
- Fit a random-intercept model and inspect convergence.
- Interpret ICC and fixed versus conditional predictions.
- Separate mixed-model inference from unseen-patient evaluation.

## R-to-Python / prerequisite bridge
`lmer(y~t+(1|id))` → `smf.mixedlm(...,groups=d["patient_id"]).fit()`; `.predict` is fixed-effect prediction, `.fittedvalues` includes fitted patient effects.

## Suggested pacing
Tasks 1–3: 18 min · Tasks 4–6: 23 min · Tasks 7–8: 11 min · Retrieval/reasoning/reflection: 8 min. Task timings are approximate; if a foundational operation needs repetition, pause there and continue later. Extensions are outside the core hour.

## Inputs and analysis contract
Start Jupyter from the repository root and select **Python (learning-python)**. Read `data/ci_telemetry/README.md` for the complete dictionary. All inputs are local, fictional teaching data: 96 patients, one implant each, 360 visits, 144 directed cells per visit. Matrix rows stimulate; columns record; contact indices are 1–12. `voltage_mv / stim_current_ua` is a synthetic kΩ V/I proxy, not a confirmed manufacturer clinical impedance algorithm. QC-valid means `ok` AND finite voltage. Mask invalid cells; unknown does not mean measured zero. No hearing, disease, or device-failure labels are available.

Write explanations in new markdown cells or print them. Exercise cells are intentionally empty. No downloads or GPU are required.

These later lessons are independently runnable: input scaffolding supplies raw representations and fixed patient indices, not model fits or engineered features. Put this setup in your first exercise cell, then add the requested code:

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from agent.course_data import load_problem, patient_partitions, NUMERIC, CATEGORICAL
problem = load_problem()
part = patient_partitions(problem)
pairs = problem["pairs"]
X = problem["X"]
y = problem["y"]
groups = problem["groups"]
matrices = problem["matrices"]   # raw current values, NaN where invalid
mask = problem["mask"]          # True where observed and usable
train, val, test = part["train"], part["val"], part["test"]
```

`train`, `val`, `test` are positions in the full prediction table. Patient counts are 54/18/24; pair counts 146/51/67. Keep IDs outside features. `target_*` fields and exact future timing are audit/outcome fields. Learn preprocessing only from the applicable training portion. Test outcomes stay reserved until Session 20. For a development-CV lesson, combine train+val, then fit all transformations inside each fold. Earlier validation is repeatedly used for learning; it is exploratory, not independent evidence.

## Concept and API reminders
A random-intercept model writes y_it = fixed mean_it + b_i + residual_it, where b_i captures stable patient-specific departures. The shared b_i creates within-patient correlation. ICC is the fraction of variance attributed to that patient intercept in this model. Random slopes extend the assumption to different patient-specific time trends.

Fixed-effect predictions average over the random-effect distribution; conditional fitted values include estimated effects for observed patients. An unseen patient's effect is unknown. REML estimates covariance after accounting for fixed effects; likelihood comparison of different fixed-effect specifications needs ML fits on identical rows. Neither approach automatically solves informative dropout or protocol/time confounding.

Imports: `import statsmodels.formula.api as smf`. Inspect `.fe_params`, `.cov_re`, `.scale`, `.converged`, and warnings. A converged optimizer does not replace scientific diagnostics.

## Task 1 — Construct the longitudinal sample (5 min)
Load visit features for training patients only. Keep complete outcome, actual time, age, array, and protocol; center age at 50. Count patients, visits, and visits per patient. Predict whether the visit count must equal four times the patient count. Explain administrative truncation in this fixture versus informative dropout in real data.

## Task 2 — Plot trajectories and protocol changes (7 min)
Plot six patient trajectories against actual elapsed months. Cross-tab scheduled window and protocol and count unique patients. Identify overlap required to distinguish a time trend from a protocol effect. Do not use scheduled month as exact elapsed time.

## Task 3 — Fit a pooled reference model (6 min)
Fit OLS with actual time, centered age, array, and protocol. Report time estimate and interval, and describe the assumption of independent residuals that repeated observations challenge. Keep this as a comparison, not the final longitudinal conclusion.

## Task 4 — Fit a random-intercept model (10 min)
Use the same formula and rows with `smf.mixedlm(formula, data, groups=data["patient_id"], re_formula="1")`; call `.fit(reml=True,method="lbfgs")`. Record `.converged`, warnings, and covariance estimates. If singularity or nonconvergence occurs, inspect scale/group counts and try another optimizer as a documented sensitivity; do not silently report an invalid fit.

## Task 5 — Interpret variance and ICC (6 min)
Extract patient-intercept variance from `.cov_re` and residual variance from `.scale`. Calculate ICC = patient variance/(patient+residual variance). Report the per-month fixed effect and interval. Explain why a high ICC suggests stable patient baselines but does not prove a useful forecast.

## Task 6 — Separate prediction types (7 min)
On observed training patients compare `.predict(data)` and `.fittedvalues`; plot conditional residuals against time. Build a grid for an unseen patient and describe why its random intercept is unknown. Predict the output length before running. Avoid supplying an observed-patient random effect to a new-patient prediction.

## Task 7 — Recover a paired descriptive contrast (7 min)
For training patients with windows 0 and 3, pivot to one row per patient and calculate month 3-minus-baseline diagonal mean. Summarize mean/median and a patient-bootstrap interval. Validate unique patient rows. Explain why this three-month contrast need not equal three times the adjusted linear-time slope.

## Task 8 — Write a longitudinal interpretation (4 min)
Report patients/visits, time association, ICC, convergence, protocol adjustment, and dropout assumptions. Explain why a random intercept alone permits neither patient-specific slopes nor causal time effects. Propose one targeted next model.

## Retrieval practice
Without copying your earlier join, match baseline and month-3 rows with `validate="one_to_one"` on patient ID and explain who is excluded.

## Professional reasoning
1. Why are learned random effects inappropriate as inputs for unseen patients?
2. What could go wrong when protocol changes almost perfectly track time?
3. Why compare different fixed-effects likelihoods using ML rather than REML?

## Self-check and reflection
Record confidence **1–5**, hardest point, one thing now explainable, what needs repetition, and workload (too light / about right / too heavy). Restart the kernel and run your attempted cells in order. Ask for review when ready; generation does not count as completion.

## Optional extension — outside the core hour
Fit a random time slope with `re_formula="~months_since_implant"`, inspect covariance and convergence. Compare fixed-effect alternatives only after fitting both with `reml=False` on identical rows.

## API references
[statsmodels OLS](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.OLS.html) · [mixed-effects models](https://www.statsmodels.org/stable/mixed_linear.html)